# LSTM Data Prep — Raw Sliding Windows for `Fault_Within_6h`

## Why this is a separate notebook from `04_lstm_classifier.ipynb`

Not a stylistic choice — a real, reproducible environment bug found by direct isolation testing. On this machine, **pandas and TensorFlow/Keras cannot coexist in the same Jupyter kernel process**: once pandas has been used at all in a kernel (even just a `groupby`), a later `model.fit()` call in *that same kernel* deadlocks — confirmed by loading pre-saved, pandas-free NumPy arrays into a kernel that had run unrelated pandas code earlier, which still hung, while the identical `fit()` call in a kernel that never touched pandas ran in seconds. This isn't fixable by reordering cells or reloading data mid-notebook — it needs a genuinely separate kernel process.

So the pipeline is split exactly where it should be anyway: **this notebook** does all the pandas-heavy work (load, build sliding windows per vehicle, split, scale) and saves the result to a small `.npz` array file. **`04_lstm_classifier.ipynb`** loads that file — no pandas import at all — and does the TensorFlow/Keras work. Same pattern the project already uses (`01b_sequence_features.ipynb` feeds `02_baseline_classifier.ipynb`/`03_sequence_classifier.ipynb` via a saved CSV), just enforced at the process level here instead of by choice.

In [1]:
import numpy as np
import pandas as pd

np.random.seed(42)

SEQ_SENSORS = ["Motor_RPM", "Motor_Torque", "Motor_Temp", "Battery_Temp"]
WINDOW_HOURS = 24  # matches the existing 24h rolling window elsewhere in the pipeline
TARGET = "Fault_Within_6h"

## 1. Load data and build raw sliding windows, per vehicle

Reuses `Fault_Within_6h` from `01b_sequence_features.ipynb` directly rather than re-deriving it — same validated target, no duplicated logic.

**Window/target alignment:** for each vehicle, a window covers `WINDOW_HOURS` (24) *consecutive raw hours*, and the label is `Fault_Within_6h` evaluated at **the hour immediately after the window ends** — i.e. the window never includes the hour it's predicting anything about, and `Fault_Within_6h` at that next hour already means "does a fault occur in the 6 hours after *that*." This is a deliberate design choice, stated explicitly rather than left implicit.

**No cross-vehicle leakage:** windows are built independently per vehicle (grouped by `user_profile`, sorted by `timestamp` first) — a window can never span two different vehicles' hours.

In [2]:
df = pd.read_csv(
    "../data/processed/driving_pattern_diagnostics_sequence_features.csv",
    parse_dates=["timestamp"],
    usecols=["timestamp", "user_profile", TARGET] + SEQ_SENSORS,
)
df = df.sort_values(["user_profile", "timestamp"]).reset_index(drop=True)


def build_windows(vehicle_df, seq_cols, target_col, window_hours, test_frac=0.2):
    """Slide a window of `window_hours` raw hours; label = target at the hour right
    after the window. Also returns a boolean train/test mask per window, using the
    SAME per-vehicle chronological cutoff logic as 02/03 (cutoff on row position,
    not on the window index) -- a window's split membership follows its TARGET row's
    chronological position."""
    raw = vehicle_df[seq_cols].to_numpy(dtype="float32")
    target = vehicle_df[target_col].to_numpy()
    n_rows = len(vehicle_df)
    cutoff = int(n_rows * (1 - test_frac))

    X, y, is_test = [], [], []
    for i in range(window_hours, n_rows):
        if pd.isna(target[i]):
            continue
        X.append(raw[i - window_hours : i])
        y.append(bool(target[i]))
        is_test.append(i >= cutoff)
    return np.array(X), np.array(y), np.array(is_test)


X_train_parts, y_train_parts, X_test_parts, y_test_parts = [], [], [], []
for profile, vdf in df.groupby("user_profile"):
    X, y, is_test = build_windows(vdf, SEQ_SENSORS, TARGET, WINDOW_HOURS)
    X_train_parts.append(X[~is_test]); y_train_parts.append(y[~is_test])
    X_test_parts.append(X[is_test]); y_test_parts.append(y[is_test])
    print(f"{profile}: {len(X)} windows total ({(~is_test).sum()} train, {is_test.sum()} test)")

X_train = np.concatenate(X_train_parts)
y_train = np.concatenate(y_train_parts)
X_test = np.concatenate(X_test_parts)
y_test = np.concatenate(y_test_parts)

print()
print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")
print(f"Train positive rate: {y_train.mean()*100:.3f}%, Test positive rate: {y_test.mean()*100:.3f}%")

daily_user: 43747 windows total (34997 train, 8750 test)
heavy_user: 43747 windows total (34997 train, 8750 test)
moderate_user: 43747 windows total (34997 train, 8750 test)
rare_user: 43747 windows total (34997 train, 8750 test)

X_train: (139988, 24, 4), X_test: (35000, 24, 4)
Train positive rate: 8.976%, Test positive rate: 9.400%


## 2. Scale (train fold only) and compute class weights (train fold only)

LSTMs are scale-sensitive, unlike the tree models — `StandardScaler` is fit on the training windows only, then applied to both splits, same leakage discipline as `02_baseline_classifier.ipynb`'s `Pipeline`. Class weights are likewise computed from `y_train` only. (`sklearn` here is fine in the same kernel as pandas — it's specifically TensorFlow that conflicts.)

In [3]:
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight

n_features = len(SEQ_SENSORS)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(
    X_train.reshape(-1, n_features)
).reshape(X_train.shape).astype("float32")
X_test_scaled = scaler.transform(
    X_test.reshape(-1, n_features)
).reshape(X_test.shape).astype("float32")

class_weight_values = compute_class_weight(class_weight="balanced", classes=np.array([False, True]), y=y_train)
class_weight = {0: class_weight_values[0], 1: class_weight_values[1]}
print("Class weights (from y_train only):", class_weight)

Class weights (from y_train only): {0: np.float64(0.5493042857254969), 1: np.float64(5.570553123756467)}


## 3. Save prepared arrays for `04_lstm_classifier.ipynb`

Saved as a single `.npz` — `04_lstm_classifier.ipynb` loads this directly and never imports pandas, by design (see the top of this notebook for why that matters here).

In [4]:
import os

os.makedirs("../data/processed", exist_ok=True)
np.savez(
    "../data/processed/lstm_fault_within_6h_windows.npz",
    X_train=X_train_scaled,
    y_train=y_train.astype("float32"),
    X_test=X_test_scaled,
    y_test=y_test.astype("float32"),
    class_weight_0=class_weight_values[0],
    class_weight_1=class_weight_values[1],
)
print(f"Saved to ../data/processed/lstm_fault_within_6h_windows.npz "
      f"({X_train_scaled.shape[0]:,} train + {X_test_scaled.shape[0]:,} test windows)")

Saved to ../data/processed/lstm_fault_within_6h_windows.npz (139,988 train + 35,000 test windows)


## Summary

- Built raw 24-hour sliding windows of `Motor_RPM`/`Motor_Torque`/`Motor_Temp`/`Battery_Temp`, per vehicle, with `Fault_Within_6h` at the hour right after each window as the target — no cross-vehicle leakage (grouped by `user_profile`), same chronological per-vehicle split methodology as `02`/`03`.
- Scaled (train-fold only) and computed class weights (train-fold only) — same leakage discipline as the tree-model pipeline.
- Saved to `data/processed/lstm_fault_within_6h_windows.npz` for `04_lstm_classifier.ipynb`, which does the actual TensorFlow/Keras work in a separate kernel process (see the top of this notebook for why that split is necessary here, not just tidy).